In [175]:
#install required libs
!pip install transformers datasets torch

In [176]:
!pip install 'accelerate>=0.26.0'

In [139]:
#Restart the kernel once the libraries are installed

In [177]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset, Dataset
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split
import torch

In [178]:
def tokenize_function(examples):
    # Here we ensure padding and truncation are applied
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "recall": recall_score(labels, predictions, average='macro'),
        "precision": precision_score(labels, predictions, average='macro'),
        "f1": f1_score(labels, predictions, average='macro')
    }

def predict_emotion(model, tokenizer, text):
    # Encoding text to suitable input format for the model
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    # Model Inference
    with torch.no_grad():  # not to keep track of gradients
        outputs = model(**inputs)

    # Probabilities (apply softmax to logits)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    probs = probabilities.detach().numpy()

    # Corresponding labels (assuming model was trained with labels in order)
    labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
    return {label: prob for label, prob in zip(labels, probs[0])}

In [179]:
# hugging face dataset on which the training dataset is based
dataset = load_dataset('emotion')

# Load datasets from local CSV files
train_dataset = load_dataset('csv', data_files='hometask_dataset.csv', split='train')
val_dataset = dataset['train'].select(range(8000, 9000))

# !!!!optional code which can be deleted once data issues are resolved!!!!!
train_df = train_dataset.to_pandas()
train_df['label'].fillna(0, inplace=True)
train_df['label'] = train_df['label'].astype(int)
train_dataset = Dataset.from_pandas(train_df)

In [208]:
# hugging face dataset on which the training dataset is based
dataset = load_dataset('emotion')

# Load datasets from local CSV files
train_dataset = load_dataset('csv', data_files='dq_output.csv', split='train')
val_dataset = dataset['train'].select(range(8000, 9000))

# !!!!optional code which can be deleted once data issues are resolved!!!!!
train_df = train_dataset.to_pandas()
train_dataset = Dataset.from_pandas(train_df)

Generating train split: 0 examples [00:00, ? examples/s]

In [209]:
model_checkpoint = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=6)

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignore

In [210]:
# Map function updated with tensor format output (applying tokenization directly on datasets)
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

In [211]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2, 
    num_train_epochs=5, 
    weight_decay=0.05,
    fp16=True,
    logging_dir='./logs',
    learning_rate=5e-5
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [212]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [221]:
# Train model
trainer.train()

/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Recall,Precision,F1
1,No log,1.072372,0.717000,0.685898,0.646797,0.646873
2,No log,1.056640,0.766000,0.702947,0.676432,0.686241
3,No log,1.226543,0.766000,0.720410,0.679603,0.694645
4,0.289024,1.540775,0.770000,0.711190,0.691761,0.696283
5,0.289024,1.552482,0.784000,0.727503,0.708529,0.713089


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=640, training_loss=0.23565891310572623, metrics={'train_runtime': 781.615, 'train_samples_per_second': 13.101, 'train_steps_per_second': 0.819, 'total_flos': 160549951242240.0, 'train_loss': 0.23565891310572623, 'epoch': 5.0})

In [222]:
# Evaluate the model
results = trainer.evaluate()
print(results)

/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

In [223]:
# You can store the trained models in different folders to be able to analyze different versions of them with the manual validation code below.
model_path = "trained_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('trained_model/tokenizer_config.json', 'trained_model/tokenizer.json')

## Manual validation

In [224]:
test_message = "I'm feeling very happy and excited today!"
test_message2 = "The news from her were astonishing. Such a great news!"
test_message3 = "If he does that again, I will break his neck!"
test_message4 = "I finally left feeling judged and ridiculed because I am intelligent"
test_message5 = "I strive to be a caring and loving partner"
test_message6 = "I stopped feeling hot, and started feeling cold"

In [225]:
# your fine-tunned model
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

In [226]:
# fine-tunned external model
classifier = pipeline("text-classification",model='bhadresh-savani/distilbert-base-uncased-emotion', return_all_scores=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Your model predictions

In [227]:
predictions = predict_emotion(loaded_model, loaded_tokenizer, test_message)
print(test_message, predictions)
predictions2 = predict_emotion(loaded_model, loaded_tokenizer, test_message2)
print(test_message2, predictions2)
predictions3 = predict_emotion(loaded_model, loaded_tokenizer, test_message3)
print(test_message3, predictions3)
predictions4 = predict_emotion(loaded_model, loaded_tokenizer, test_message4)
print(test_message4, predictions4)
predictions5 = predict_emotion(loaded_model, loaded_tokenizer, test_message5)
print(test_message5, predictions5)
predictions6 = predict_emotion(loaded_model, loaded_tokenizer, test_message6)
print(test_message6, predictions6)

I'm feeling very happy and excited today! {'sadness': 1.0413129e-09, 'joy': 0.99999964, 'love': 2.2617277e-08, 'anger': 3.3621113e-09, 'fear': 1.1015776e-09, 'surprise': 3.4781345e-07}
The news from her were astonishing. Such a great news! {'sadness': 8.253803e-10, 'joy': 2.2374977e-08, 'love': 3.7189165e-11, 'anger': 8.143683e-10, 'fear': 4.591343e-07, 'surprise': 0.9999995}
If he does that again, I will break his neck! {'sadness': 9.795636e-05, 'joy': 7.894864e-06, 'love': 3.0428122e-08, 'anger': 0.6168482, 'fear': 0.3827797, 'surprise': 0.00026630194}
I finally left feeling judged and ridiculed because I am intelligent {'sadness': 7.837149e-13, 'joy': 1.0, 'love': 4.8936384e-12, 'anger': 3.2384143e-10, 'fear': 1.4909325e-13, 'surprise': 7.9140666e-11}
I strive to be a caring and loving partner {'sadness': 7.7082965e-05, 'joy': 0.050186045, 'love': 0.9497368, 'anger': 3.516565e-08, 'fear': 3.6760404e-08, 'surprise': 5.4482534e-09}
I stopped feeling hot, and started feeling cold {'sad

## External model predictions

In [228]:
prediction = classifier(test_message)
prediction2 = classifier(test_message2)
prediction3 = classifier(test_message3)
prediction4 = classifier(test_message4)
prediction5 = classifier(test_message5)
prediction6 = classifier(test_message6)
print(test_message, prediction)
print(test_message2, prediction2)
print(test_message3, prediction3)
print(test_message4, prediction4)
print(test_message5, prediction5)
print(test_message6, prediction6) #-> 'bhadresh-savani/distilbert-base-uncased-emotion' gives incorrect label, try vise versa hot/cold

I'm feeling very happy and excited today! [{'label': 'joy', 'score': 0.999071478843689}]
The news from her were astonishing. Such a great news! [{'label': 'surprise', 'score': 0.859711229801178}]
If he does that again, I will break his neck! [{'label': 'anger', 'score': 0.6476961970329285}]
I finally left feeling judged and ridiculed because I am intelligent [{'label': 'joy', 'score': 0.9982800483703613}]
I strive to be a caring and loving partner [{'label': 'love', 'score': 0.99539715051651}]
I stopped feeling hot, and started feeling cold [{'label': 'love', 'score': 0.9912281036376953}]
